# VK- EKF Implementation



In the VKSSM, each agent has:

* an $x$-position
* a $y$-position
* a heading $\theta$

So for $N$ agents, the hidden state at time $t$ is:
$$
z_t = (x_1,y_1,\theta_1,; x_2,y_2,\theta_2,; \dots,; x_N,y_N,\theta_N).
$$

That means the EKF works with one  vector of length $3N$.

Our observation model only measures positions, not headings. So the observed data is:
$$
y_t = (x_1,y_1,; x_2,y_2,; \dots,; x_N,y_N),
$$
which has length $2N$.

The following code makes sure that the hidden state and observations are of the correct form.



The function `generate_noisy_observations_from_pos(pos, obs_std, seed=123)` generates noisy observations.

This takes the true simulated positions and adds Gaussian noise.

So if the true positions are $(x_i(t), y_i(t)),$
the observed positions become
$$
x_i^{obs}(t)=x_i(t)+\varepsilon_{x,i,t},\qquad
y_i^{obs}(t)=y_i(t)+\varepsilon_{y,i,t},
$$
with
$$
\varepsilon_{x,i,t},\varepsilon_{y,i,t}\sim \mathcal N(0,\sigma_{obs}^2).
$$



### State Conversion

In [ ]:


# State conversion

def wrap_angle(theta):
    return (theta + np.pi) % (2 * np.pi) - np.pi

def pack_state(pos, theta):
    """
    pos: (N,2), theta: (N,)
    returns z: (3N,)
    ordering [x1,y1,th1, x2,y2,th2, ...]
    """
    N = pos.shape[0]
    z = np.zeros(3 * N)
    z[0::3] = pos[:, 0]
    z[1::3] = pos[:, 1]
    z[2::3] = wrap_angle(theta)
    return z


def unpack_state(z):
    N = len(z) // 3
    pos = np.zeros((N, 2))
    pos[:, 0] = z[0::3]
    pos[:, 1] = z[1::3]
    theta = wrap_angle(z[2::3])
    return pos, theta


### Observation Model 

In [ ]:



# Observation model

def observation_h(z):
    """
    Observe positions only.
    """
    pos, _ = unpack_state(z)
    y = np.zeros(2 * pos.shape[0])
    y[0::2] = pos[:, 0]
    y[1::2] = pos[:, 1]
    return y


def observation_matrix_H(N):
    H = np.zeros((2 * N, 3 * N))
    for i in range(N):
        H[2 * i, 3 * i] = 1.0
        H[2 * i + 1, 3 * i + 1] = 1.0
    return H


def generate_noisy_observations_from_pos(pos, obs_std, seed=123):
    """
    pos: (T+1, N, 2)
    returns obs: (T+1, 2N)
    """
    rng = np.random.default_rng(seed)
    T_plus_1, N, _ = pos.shape
    obs = np.zeros((T_plus_1, 2 * N))
    for t in range(T_plus_1):
        obs[t, 0::2] = pos[t, :, 0] + obs_std * rng.standard_normal(N)
        obs[t, 1::2] = pos[t, :, 1] + obs_std * rng.standard_normal(N)
    return obs

## Fixed Interaction Graph



We construct the interaction matrix $W$ from the current agent positions using a hard-radius rule under periodic boundary conditions.

Each entry $W_{ij}$ indicates whether agent $j$ influences agent $i$:
- $W_{ij} = 1$ if agent \(j\) lies within radius $R$ of agent $i$,
- $W_{ij} = 0$ otherwise.

In the EKF implementation, this matrix is treated as locally fixed during linearisation. That is, we do not differentiate $W$ with respect to positions, since the hard-radius graph is non-smooth.

In [ ]:
def vk_state_transition(z, params):
    """
    One step deterministic state update for the Vicsek–Kuramoto model.

    State ordering:
        z = [x1, y1, theta1, x2, y2, theta2, ..., xN, yN, thetaN]

    The update is:
        1. compute the interaction matrix from current positions,
        2. update headings,
        3. move each agent using the updated heading,
        4. wrap positions and headings back to the periodic domain.
    """
    pos, theta = unpack_state(z)

    # Build the current interaction matrix from the agent positions.
    W = build_weight_matrix_radius(
        pos, params.R, params.L,
        include_self=params.include_self,
        weight_mode=params.weight_mode
    )

    # diff[i, j] = theta_j - theta_i
    diff = theta[None, :] - theta[:, None]

    # Alignment drift for each agent:
    # drift_i = sum_j W_ij sin(theta_j - theta_i)
    drift = (W * np.sin(diff)).sum(axis=1)

    # Update headings and wrap back to [-pi, pi).
    theta_next = wrap_angle(theta + params.beta * params.dt * drift)

    # Move each agent forward using its updated heading.
    step = params.v * params.dt * np.column_stack([
        np.cos(theta_next),
        np.sin(theta_next)
    ])
    pos_next = wrap_box(pos + step, params.L)

    return pack_state(pos_next, theta_next)


def vk_jacobian_F_approx(z, params):
    """
    Approximate Jacobian of the state transition map used in the EKF.

    This linearisation treats the interaction matrix W as fixed at the current
    state. In particular, it does not differentiate through the neighbour set
    with respect to position, which is appropriate here because the hard-radius
    interaction rule is not smooth.
    """
    pos, theta = unpack_state(z)
    N = pos.shape[0]

    # Evaluate the neighbour matrix at the current state.
    W = build_weight_matrix_radius(
        pos, params.R, params.L,
        include_self=params.include_self,
        weight_mode=params.weight_mode
    )

    diff = theta[None, :] - theta[:, None]
    cos_diff = np.cos(diff)

    # Start from the identity map, then overwrite the nontrivial blocks.
    F = np.eye(3 * N)

    # G stores derivatives of the heading update with respect to headings:
    # G[i, j] = d(theta_next_i) / d(theta_j)
    G = np.zeros((N, N))
    for i in range(N):
        G[i, i] = 1.0 - params.beta * params.dt * np.sum(W[i] * cos_diff[i])
        for j in range(N):
            if j != i:
                G[i, j] = params.beta * params.dt * W[i, j] * cos_diff[i, j]

    # Insert the heading-to-heading block into the full Jacobian.
    for i in range(N):
        for j in range(N):
            F[3 * i + 2, 3 * j + 2] = G[i, j]

    # Position update depends on theta_next, so apply the chain rule:
    # x_i' = x_i + v dt cos(theta_next_i)
    # y_i' = y_i + v dt sin(theta_next_i)
    for i in range(N):
        ix = 3 * i
        iy = 3 * i + 1

        theta_next_i = theta[i] + params.beta * params.dt * np.sum(W[i] * np.sin(diff[i]))
        theta_next_i = wrap_angle(theta_next_i)

        for j in range(N):
            dtheta_next_i_dtheta_j = G[i, j]

            F[ix, 3 * j + 2] += (
                params.v * params.dt * (-np.sin(theta_next_i)) * dtheta_next_i_dtheta_j
            )
            F[iy, 3 * j + 2] += (
                params.v * params.dt * np.cos(theta_next_i) * dtheta_next_i_dtheta_j
            )

    return F